In [1]:
!pip -q install rpy2
%load_ext rpy2.ipython



[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
%%R
options(repos = c(CRAN = "https://cloud.r-project.org"))
if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, ggplot2, dplyr, lubridate)

Loading required package: pacman


In [3]:

%%R
# 1. Read the raw text files
v1996 = read_tsv('../data/HCRIS_Data_v1996.txt')
v2010 = read_tsv('../data/HCRIS_Data_v2010.txt')

# 2. Add missing columns to the 1996 format so it matches 2010
v1996_updated = v1996 %>%
  mutate(hvbp_payment = NA, hrrp_payment = NA)

# 3. Use bind_rows (this handles the "numbers of columns" error)
final.hcris = bind_rows(v1996_updated, v2010) %>%
  mutate(fy_end = mdy(fy_end), 
         fy_start = mdy(fy_start),
         date_processed = mdy(date_processed), 
         date_created = mdy(date_created),
         tot_discounts = abs(tot_discounts), 
         # Ensure hrrp_payment isn't NA before doing math
         hrrp_payment = abs(if_else(is.na(hrrp_payment), 0, hrrp_payment))) %>%
  mutate(fyear = year(fy_end)) %>%
  # Add the report count here so it's ready for your graph
  add_count(provider_number, fyear, name="total_reports") %>%
  arrange(provider_number, fyear) %>%
  select(-any_of("year"))

# 4. Check that it worked
print(paste("Rows in dataset:", nrow(final.hcris)))
head(final.hcris)

Rows: 78508 Columns: 47
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): provider_number, fy_start, fy_end, date_processed, date_created, d...
dbl (34): report, status, year, beds, tot_charges, net_pat_rev, tot_discount...
lgl  (1): npi

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 63852 Columns: 48
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): provider_number, fy_start, fy_end, date_processed, date_created, d...
dbl (35): report, status, year, beds, tot_charges, tot_discounts, net_pat_re...
lgl  (1): npi

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
[1] "Rows in dataset: 142360"
# A tibble: 6 × 54
  report provider_number npi   fy_start   fy_end   

In [4]:
%%R
final.hcris = final.hcris %>% add_count(provider_number, fyear, name="total_reports")

final.hcris = final.hcris %>% 
  group_by(provider_number, fyear) %>%
  mutate(report_number=row_number())

unique.hcris1 = final.hcris %>%
  filter(total_reports==1) %>%
  select(-report, -total_reports, -report_number, -npi, -status) %>%
  mutate(source='unique reports')

duplicate.hcris = final.hcris %>%
  filter(total_reports>1) %>%
  mutate(time_diff=fy_end-fy_start) %>%
  group_by(provider_number, fyear) %>%
  mutate(total_days=sum(time_diff))

unique.hcris2 = duplicate.hcris %>%
  filter(total_days<370) %>%
  group_by(provider_number, fyear) %>%
  mutate(hrrp_payment=if_else(is.na(hrrp_payment), 0, hrrp_payment),
         hvbp_payment=if_else(is.na(hvbp_payment), 0, hvbp_payment)) %>%
  summarize(beds=max(beds), tot_charges=sum(tot_charges), tot_discounts=sum(tot_discounts),
            tot_operating_exp=sum(tot_operating_exp), ip_charges=sum(ip_charges),
            icu_charges=sum(icu_charges), ancillary_charges=sum(ancillary_charges),
            tot_discharges=sum(tot_discharges), mcare_discharges=sum(mcare_discharges),
            mcaid_discharges=sum(mcaid_discharges), tot_mcare_payment=sum(tot_mcare_payment),
            secondary_mcare_payment=sum(secondary_mcare_payment), hvbp_payment=sum(hvbp_payment),
            hrrp_payment=sum(hrrp_payment), fy_start=min(fy_start), fy_end=max(fy_end),
            date_processed=max(date_processed), date_created=min(date_created), 
            street=first(street), city=first(city), state=first(state),
            zip=first(zip), county=first(county), .groups = 'drop') %>%
  mutate(source='total for year')

duplicate.hcris2 = duplicate.hcris %>%
  filter(total_days>=370) %>%
  mutate(max_days=max(time_diff), max_date=max(fy_end))

unique.hcris3 = duplicate.hcris2 %>%
  filter(max_days==time_diff, time_diff>360, max_date==fy_end) %>%
  select(-report, -total_reports, -report_number, -npi, -status, -max_days, -time_diff, -total_days, -max_date) %>%
  mutate(source='primary report')

duplicate.hcris3 = anti_join(duplicate.hcris2, unique.hcris3, by=c("provider_number", "fyear")) %>%
  mutate(total_days=as.integer(total_days), time_diff=as.integer(time_diff)) %>%
  mutate(across(c(tot_charges, tot_discounts, tot_operating_exp, ip_charges, 
                  icu_charges, ancillary_charges, tot_discharges, mcare_discharges, 
                  mcaid_discharges, tot_mcare_payment, secondary_mcare_payment, 
                  hvbp_payment, hrrp_payment), ~ . * (time_diff/total_days)))

unique.hcris4 = duplicate.hcris3 %>%
  group_by(provider_number, fyear) %>%
  mutate(hrrp_payment=if_else(is.na(hrrp_payment), 0, hrrp_payment),
         hvbp_payment=if_else(is.na(hvbp_payment), 0, hvbp_payment)) %>%
  summarize(beds=max(beds), tot_charges=sum(tot_charges), tot_discounts=sum(tot_discounts),
            tot_operating_exp=sum(tot_operating_exp), ip_charges=sum(ip_charges),
            icu_charges=sum(icu_charges), ancillary_charges=sum(ancillary_charges),
            tot_discharges=sum(tot_discharges), mcare_discharges=sum(mcare_discharges),
            mcaid_discharges=sum(mcaid_discharges), tot_mcare_payment=sum(tot_mcare_payment),
            secondary_mcare_payment=sum(secondary_mcare_payment), hvbp_payment=sum(hvbp_payment),
            hrrp_payment=sum(hrrp_payment), fy_start=min(fy_start), fy_end=max(fy_end),
            date_processed=max(date_processed), date_created=min(date_created), 
            street=first(street), city=first(city), state=first(state),
            zip=first(zip), county=first(county), .groups = 'drop') %>%
  mutate(source='weighted_average')

In [5]:
%%R
final.hcris.data = rbind(unique.hcris1, unique.hcris2, unique.hcris3, unique.hcris4) %>%
  rename(year=fyear) %>%
  arrange(provider_number, year)

if (!dir.exists("../data")) dir.create("../data")

write_csv(final.hcris.data, '../data/final_hcris_data.csv')